# Need for Autograd

Consider the following operations:

- \( y = x^2 \)
- \( z = \sin(y) \)
- \( u = e^z \)

To compute the derivative of \(u\) with respect to \(x\):

:contentReference[oaicite:0]{index=0}

- This is done using the **Chain Rule**.
- For simple expressions, manual differentiation is manageable.
- However, deep neural networks may contain millions of parameters and thousands of operations.
- Writing code to compute gradients manually at every step becomes difficult and error-prone.

**Solution:** PyTorch's **Autograd**.

---

# Autograd

- **Autograd** is a core component of PyTorch that provides **automatic differentiation** for tensor operations.
- It automatically computes gradients required for training deep learning models.
- Autograd tracks all operations performed on tensors and builds a **computation graph**.
- During the backward pass, it uses the **Chain Rule** to calculate gradients automatically.

### Why is Autograd Important?

- Eliminates manual gradient calculations.
- Simplifies backpropagation.
- Enables efficient training of neural networks.
- Works seamlessly with optimization algorithms such as:
  - Gradient Descent (GD)
  - Stochastic Gradient Descent (SGD)
  - Adam
  - RMSProp

### In Short

> Autograd automatically computes gradients for tensor operations, making neural network training efficient and practical.

In [1]:
import torch

In [8]:
x = torch.tensor(3.0, requires_grad = True) #pytorch knows that we need to create it's derivativ
y = x ** 2
print(x)
print(y)
y.backward()
print(x.grad)

tensor(3., requires_grad=True)
tensor(9., grad_fn=<PowBackward0>)
tensor(6.)


## Example

```python
x = torch.tensor(3.0, requires_grad=True)

y = x ** 2

y.backward()

print(x.grad)
```

### Computation Graph

```text
x = 3.0
│
│ square (x²)
▼
y = 9.0
```

Or more formally:

```text
x
│
│ x²
▼
y
```

### Working

#### Step 1: Create Tensor

```python
x = torch.tensor(3.0, requires_grad=True)
```

- `requires_grad=True` tells PyTorch to track all operations on `x`.
- PyTorch starts building a computation graph.

---

#### Step 2: Perform Operation

```python
y = x ** 2
```

- PyTorch records the operation:
    
    ```text
    y = x²
    ```
    
- The computation graph now contains:
    
    ```text
    x ──► square ──► y
    ```

---

#### Step 3: Backward Pass

```python
y.backward()
```

- PyTorch traverses the graph in reverse.
- Computes:



- Since `x = 3`:



---

#### Step 4: Access Gradient

```python
print(x.grad)
```

Output:

```text
tensor(6.)
```

- The computed gradient is stored in `x.grad`.

### Summary

```text
Forward Pass:
x = 3
y = x² = 9

Backward Pass:
dy/dx = 2x
dy/dx = 6

Stored in:
x.grad = 6
```

In [10]:
# - y = x2
# - z = sin(y)
# - dz/dx = dz/dy * dy/dx

x = torch.tensor(4.0, requires_grad = True)
y = x ** 2
z = torch.sin(y)
z.backward()
print(x.grad)

tensor(-7.6613)


## Computation Graph

```text
x = 3
│
│ square (x²)
▼
y = 9
│
│ sin(y)
▼
z = sin(9)
```
```text
Forward:
x → y = x² → z = sin(y)

Backward:
dz/dx = dz/dy × dy/dx

Gradient stored in:
x.grad
```

In [13]:
print(y.grad) # at intermediate nodes no grads

None


/tmp/ipykernel_4995/432274887.py:1: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:494.)
  print(y.grad) # at intermediate nodes no grads


## real use case of forward pass and back pass

# Logistic Regression / Single Neuron

```text
          w
X  ─────────────►  z = wx + b
                     │
                     ▼
                ┌─────────┐
                │ Sigmoid │
                └─────────┘
                     │
                     ▼
                  ŷ (y_hat)
                     │
                     ▼
                   Loss
```

---

## 1. Linear Transformation

:contentReference[oaicite:0]{index=0}

- `x` → Input Feature
- `w` → Weight
- `b` → Bias
- `z` → Linear Output

---

## 2. Activation Function (Sigmoid)



- Converts output to a probability between `0` and `1`.

---

## 3. Loss Function (Binary Cross Entropy)


- Measures prediction error.
- Lower loss ⇒ Better predictions.

---

## Computation Graph

```text
x
│
├──► (× w)
│
└──► (+ b)
         │
         ▼
      z = wx+b
         │
         ▼
      Sigmoid
         │
         ▼
      y_hat
         │
         ▼
       Loss
```

In [14]:
X = torch.tensor(6.7)
y = torch.tensor(0.0)

In [15]:
w = torch.tensor(1.0, requires_grad = True)
b = torch.tensor(0.0, requires_grad = True)

In [16]:
print(w)
print(b)

tensor(1., requires_grad=True)
tensor(0., requires_grad=True)


In [19]:
z = w*X + b

In [20]:
z

tensor(6.7000, grad_fn=<AddBackward0>)

In [21]:
y_pred = torch.sigmoid(z)

In [29]:
import torch

def binary_cross_entropy(y_true, y_pred):
    return -(y_true * torch.log(y_pred) +
             (1 - y_true) * torch.log(1 - y_pred))

In [30]:
loss = binary_cross_entropy(y, y_pred)

In [31]:
loss

tensor(6.7012, grad_fn=<NegBackward0>)

In [32]:
loss.backward()

In [33]:
print(w.grad)
print(b.grad)

tensor(6.6918)
tensor(0.9988)


In [35]:
# Gradient Accumulation Problem

import torch

x = torch.tensor(2.0, requires_grad=True)

# First backward pass
y = x ** 2
y.backward()

print("After 1st backward:", x.grad)

# Second backward pass
y = x ** 2
y.backward()

print("After 2nd backward:", x.grad)

### Output

# ```text
# After 1st backward: tensor(4.)
# After 2nd backward: tensor(8.)
# ```



# Solution: Clear Gradients


import torch

x = torch.tensor(2.0, requires_grad=True)

# First backward pass
y = x ** 2
y.backward()

print("After 1st backward:", x.grad)

# Clear gradients
x.grad.zero_()

# Second backward pass
y = x ** 2
y.backward()

print("After clearing + backward:", x.grad)


### Output

# ```text
# After 1st backward: tensor(4.)
# After clearing + backward: tensor(4.)
# ```

After 1st backward: tensor(4.)
After 2nd backward: tensor(8.)
After 1st backward: tensor(4.)
After clearing + backward: tensor(4.)
